# Climate disclosure classification — evidence and decision notebook
**Simone Rossi · revised 2 November 2026 · protocol v2**

## Stakeholder briefing — plain language
This tool sorts English climate-related paragraphs into **risk**, **neutral**, and **opportunity** so an analyst can prioritize reading. It classifies what a company says; it does **not** verify emissions, climate performance, greenwashing, or investment quality.

**Current recommendation:** use as an analyst-assistance prototype with human review. A fresh controlled transformer comparison is required before claiming that the transformer is the better choice. The executed results and decision checks below update from saved predictions, rather than manually maintained scores.

The original saved run reported transformer accuracy **85.0%** and macro F1 **0.8470**, versus classical accuracy **76.0%** and macro F1 **0.7547**. That is a **9.23 percentage-point** F1 difference, but training budgets differed and the same small test set was explored repeatedly. It is historical evidence, not proof of a general improvement. The original table's 86% / 0.8570 has been corrected.

Read **Results for stakeholders**, **Decision and guardrails**, and **Acceptance-criteria evidence** for the decision. Technical cells explain how to reproduce it. Historical outputs are preserved in [the original submission](evidence/original_submission.ipynb) and [extracted metrics](evidence/legacy_metrics.json).

## 1. Context, success criteria, and boundaries
**User and decision:** a sustainability research team needs useful triage, protection against missed risk paragraphs, a manageable review queue, and affordable inference. Missing a risk paragraph is more consequential than sending an extra paragraph to a reviewer.

**Proposed gates, set before the revised run (assumptions, not stakeholder-approved targets):** macro F1 ≥ 0.80, risk recall ≥ 0.85, and inference ≤ 100 ms/paragraph on the recorded target device. Prefer DistilBERT only if its paired test F1 gain is ≥ 0.03 and its 95% paired bootstrap interval excludes zero, with no loss of risk recall; otherwise prefer the simpler qualifying classical model. A review-routing policy must retain ≥ 50% of paragraphs with ≥ 95% observed accuracy, at least 30 retained validation rows, and confirmation on test. These are pilot screens, not statistical safety guarantees.

**Metric meaning:** macro F1 gives each class equal importance; risk recall measures how many labelled risk paragraphs are found. Accuracy is the overall correct share. Macro average precision is one-vs-rest **average precision**, not trapezoidal PR-AUC. Micro F1 equals accuracy for this single-label task. Probability scores are uncalibrated ranking signals, not verified likelihoods of correctness.

### Key assumptions and validity threats
- Paragraphs in the public dataset approximate the pilot workload. They do not establish performance on new issuers, years, languages, or full reports.
- The source gives **risk / neutral / opportunity**, not general negative / neutral / positive sentiment. A risk-mitigation promise need not be an opportunity.
- Shared issuer boilerplate and near duplicates can inflate random-split scores. Exact normalized duplicates are checked below; a nearest-neighbor diagnostic quantifies residual similarity. Issuer IDs and dates are unavailable, so issuer-disjoint / temporal generalization remains untested.
- Ambiguous mixed risk/opportunity paragraphs may have disputable labels. Errors are examples of disagreement with the dataset, not proof of annotation errors.
- The test set is now inspected for reporting. Changes motivated by these errors require a **new untouched evaluation set**, not repeated test tuning.

## 2. Reproduce locally or on Colab
The notebook is self-contained: uploading this `.ipynb` alone to Colab is sufficient. It embeds the default config, public-data checksums, and historical evidence. Companion files make review easier: [core environment](requirements.txt), [GPU environment](requirements-transformer.txt), [run configuration](configs/run_config.json), and [data manifest](data/manifest.json).

**Colab:** select a GPU runtime (T4 is sufficient), run the setup cell once, then **restart the session** after installation and Run all. The bootstrap writes pinned requirements and installs them only when the installed versions differ. Colab's base image, driver, GPU type and transitive packages can vary; every run records them. Download the run ZIP at the end before the runtime expires. No Google Drive mount or private token is required.

**VS Code:** with Google's Colab extension, use **Select Kernel → Colab → Auto Connect**, then choose a GPU. [Official extension instructions](https://marketplace.visualstudio.com/items?itemName=Google.colab). [Colab import and runtime guidance](https://research.google.com/colaboratory/faq.html).

**Local baseline:** `python -m pip install -r requirements.txt`, then `python scripts/run_notebook.py --mode baseline`. Full GPU run: install `requirements-transformer.txt`, then `python scripts/run_notebook.py --mode full`. An explicit `full` run fails if CUDA is unavailable. `auto` completes the classical evidence and visibly marks transformer checks pending on CPU.

Reproducible does not mean bit-identical across devices: fixed data hashes, row IDs, seeds, package versions and model commits make the experiment traceable. Deterministic PyTorch algorithms, FP32 and seeded initialization reduce variation. Bootstrap intervals cover test-sampling uncertainty, **not** training variation, issuer clustering, label uncertainty, or domain shift. The GPU profile runs seeds 42, 43 and 44 and reports them separately.

In [1]:
import os, sys, json, subprocess, importlib.metadata
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
ROOT = Path.cwd()
CORE_REQUIREMENTS = '# Python 3.13.5 reference; Python 3.12/3.13 supported (record exact runtime per run).\nnumpy==2.1.3\npandas==2.2.3\nscipy==1.15.3\nscikit-learn==1.6.1\nmatplotlib==3.10.0\npyarrow==19.0.0\njoblib==1.4.2\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel==6.29.5\nnbconvert==7.16.6\n'
GPU_REQUIREMENTS = "-r requirements.txt\n# Separate GPU profile; no transformer training is started on CPU by default.\ntorch==2.8.0\ntransformers==4.57.1\ndatasets==4.3.0\naccelerate==1.11.0\nhuggingface-hub==0.36.0\ntokenizers==0.22.1\nsafetensors==0.6.2\n# Keep Colab's optional vision/audio packages ABI-compatible with torch.\ntorchvision==0.23.0\ntorchaudio==2.8.0\n"

if IN_COLAB:
    (ROOT / 'requirements.txt').write_text(CORE_REQUIREMENTS)
    (ROOT / 'requirements-transformer.txt').write_text(GPU_REQUIREMENTS)
    wanted = [line for line in (CORE_REQUIREMENTS + GPU_REQUIREMENTS).splitlines()
              if '==' in line and not line.startswith('#')]
    mismatch = []
    for spec in wanted:
        package, version = spec.split('==')
        try:
            current = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            current = None
        if current != version:
            mismatch.append(spec)
    if mismatch:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               '-r', str(ROOT / 'requirements-transformer.txt')])
        raise RuntimeError('Pinned packages installed. Restart the Colab session, then Run all.')
print('Runtime:', 'Colab' if IN_COLAB else 'local / remote Jupyter', sys.version.split()[0])

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/content/requirements-transformer.txt']' returned non-zero exit status 1.

### 2.1 Configuration and run record
Each execution creates its own `artifacts/<UTC timestamp>_<config hash>/`. It saves the effective config, package snapshot, hardware, splits, model files, predictions, tables, figures and a checksum manifest. A complete manifest is written only in the final cell. `latest_run.json` points to that run. The environment variable `NLP_CONFIG_PATH` selects an alternative config; the effective copy is always saved.

In [ ]:
import hashlib, platform, random, re, time, shutil, zipfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score, average_precision_score,
                             confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize
import joblib
DEFAULT_CONFIG = {'protocol_version': '2.0', 'seed': 42, 'validation_fraction': 0.2, 'dataset_revision': 'f261b1fd1af11195db3bdfd809a37a85e0d9fbc2', 'model_id': 'distilbert/distilbert-base-uncased', 'model_revision': '12040accade4e8a0f71eabdb258fecc2e7e948be', 'baseline': {'min_df': 5, 'max_features': 1000, 'C_grid': [0.1, 1.0, 10.0], 'alpha': 1.0, 'max_iter': 1000}, 'transformer': {'epochs': 3, 'learning_rate': 2e-05, 'batch_size': 16, 'max_length': 128, 'weight_decay': 0.01, 'warmup_ratio': 0.0, 'fp16': False, 'early_stopping_patience': 1, 'seeds': [42, 43, 44]}, 'bootstrap_resamples': 2000, 'decision': {'min_macro_f1': 0.8, 'min_risk_recall': 0.85, 'min_transformer_gain': 0.03, 'min_retained_accuracy': 0.95, 'min_coverage': 0.5, 'min_validation_retained_n': 30, 'max_inference_ms_per_paragraph': 100.0}, 'routing_threshold_grid': [0.5, 0.6, 0.7, 0.8, 0.9, 0.95], 'run_transformer': 'auto_cuda_only', 'run_pos': False}
DATA_MANIFEST = {'dataset': 'climatebert/climate_sentiment', 'revision': 'f261b1fd1af11195db3bdfd809a37a85e0d9fbc2', 'files': {'test-00000-of-00001-3f9f7af4f5914b8e.parquet': {'sha256': '47bc8ce57d5d975628063362827543df9382511e1a6d501896bf3a678a36fc75', 'bytes': 100977}, 'train-00000-of-00001-04b49ae22f595095.parquet': {'sha256': '709f6dd5b265e9de31357c112ce7673ee3522e364e7e0712c649216f83a4499b', 'bytes': 272661}}, 'label_names': ['risk', 'neutral', 'opportunity'], 'license': 'CC-BY-NC-SA-4.0', 'source_card': 'https://huggingface.co/datasets/climatebert/climate_sentiment/blob/f261b1fd1af11195db3bdfd809a37a85e0d9fbc2/README.md'}
LEGACY = {'status': 'historical_saved_output_not_rerun', 'source_file': 'evidence/original_submission.ipynb', 'source_sha256': '92c81b95689eea7816943fd7c513652506c5a4b1ae7356caa037378bb57a6ba6', 'protocol': 'Original 800/100/100 split of official train; classical refit on 900, transformer trained on 800; not a controlled architecture comparison.', 'baseline': {'accuracy': 0.76, 'macro_f1': 0.7547, 'macro_average_precision': 0.8974}, 'transformer': {'test_loss': 0.45611652731895447, 'test_accuracy': 0.85, 'test_macro_f1': 0.8470351354072285, 'test_runtime': 20.5996, 'test_samples_per_second': 4.854, 'test_steps_per_second': 0.34}, 'training': {'runtime_seconds': 1886.7016, 'validation_loss': [0.5990496873855591, 0.4696476459503174, 0.4517006576061249], 'validation_macro_f1': [0.7950554134697357, 0.8295284524908905, 0.8295284524908905]}, 'limitations': ['No saved checkpoint or per-row transformer prediction file supplied.', 'Environment not locked; kernel state not recoverable.', 'Saved markdown table incorrectly reported accuracy 0.86 and macro F1 0.8570.', 'Historical metrics are rounded where only formatted output is available.']}

config_path = Path(os.environ.get('NLP_CONFIG_PATH', ROOT / 'configs/run_config.json'))
CONFIG = json.loads(config_path.read_text()) if config_path.exists() else DEFAULT_CONFIG.copy()
assert CONFIG['dataset_revision'] == DATA_MANIFEST['revision'], 'Update the checksummed data manifest with a new revision.'
MODE = os.environ.get('NLP_RUN_MODE', 'auto')
assert MODE in ('auto', 'baseline', 'full')
CONFIG['execution_mode'] = MODE
CONFIG['run_pos'] = os.environ.get('NLP_RUN_POS', str(CONFIG['run_pos'])).lower() == 'true'
SEED = CONFIG['seed']
random.seed(SEED)
np.random.seed(SEED)
LABELS = [0, 1, 2]
NAMES = ['risk', 'neutral', 'opportunity']
COLORS = ['#b54736', '#526b80', '#167c70']
plt.rcParams.update({'figure.figsize': (10, 4.8), 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'savefig.facecolor': 'white', 'axes.labelcolor': '#202830', 'text.color': '#202830'})
pd.set_option('display.max_colwidth', 110)
config_hash = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + config_hash[:10]
RUN_DIR = ROOT / 'artifacts' / RUN_ID
for directory in ['tables', 'figures', 'models', 'predictions', 'data']:
    (RUN_DIR / directory).mkdir(parents=True, exist_ok=True)
def save_json(name, obj):
    (RUN_DIR / name).write_text(json.dumps(obj, indent=2, default=str, allow_nan=False) + '\n')
def save_table(name, frame):
    frame.to_csv(RUN_DIR / 'tables' / (name + '.csv'), index=False)
def figure(name):
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'figures' / (name + '.png'), dpi=150, bbox_inches='tight')
    display(Image(filename=str(RUN_DIR / 'figures' / (name + '.png'))))
    plt.close()
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
save_json('run_config.json', CONFIG)
save_json('source_manifest.json', DATA_MANIFEST)
save_json('legacy_metrics.json', LEGACY)
(RUN_DIR / 'requirements.txt').write_text(CORE_REQUIREMENTS)
(RUN_DIR / 'requirements-transformer.txt').write_text(GPU_REQUIREMENTS)
packages = sorted(f'{d.metadata["Name"]}=={d.version}' for d in importlib.metadata.distributions() if d.metadata['Name'])
(RUN_DIR / 'environment.freeze.txt').write_text('\n'.join(packages) + '\n')
ENVIRONMENT = {'python':sys.version, 'platform':platform.platform(), 'processor':platform.processor(),
               'colab':IN_COLAB, 'config_sha256':config_hash, 'run_id':RUN_ID,
               'thread_settings':{k:os.environ.get(k) for k in ['OMP_NUM_THREADS','MKL_NUM_THREADS','PYTHONHASHSEED']}}
save_json('environment.json', ENVIRONMENT)
# Save exactly the cells used, without outputs, for source-level reproduction.
notebook_path = ROOT / 'NLP_Simone_Rossi.ipynb'
if notebook_path.exists():
    source_notebook = json.loads(notebook_path.read_text())
    for cell in source_notebook['cells']:
        if cell['cell_type'] == 'code':
            cell['outputs'], cell['execution_count'] = [], None
    save_json('notebook_source.ipynb', source_notebook)
print('Run:', RUN_ID, '| mode:', MODE)

## 3. Data, quality checks, and controlled splits
[ClimateSentiment source card](https://huggingface.co/datasets/climatebert/climate_sentiment) describes expert-labelled paragraphs from corporate reports. The exact revision and SHA-256 checksums are saved in the manifest. Public data is cached under `data/`; a hash mismatch fails rather than silently downloading a different version.

**Revised protocol:** use the official 1,000 training paragraphs for an 800/200 stratified train/validation split; keep the official 320 test paragraphs for the selected classical model and prespecified transformer seeds. Vocabulary, preprocessing decisions, model selection and routing thresholds use development data only. Neither family is refit on validation. The original 800/100/100 experiment remains a separately labelled historical record.

In [ ]:
from urllib.request import urlretrieve
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
for filename, info in DATA_MANIFEST['files'].items():
    path = DATA_DIR / filename
    if not path.exists():
        url = f"https://huggingface.co/datasets/{DATA_MANIFEST['dataset']}/resolve/{DATA_MANIFEST['revision']}/data/{filename}"
        urlretrieve(url, path)
    assert sha256(path) == info['sha256'], f'Checksum mismatch: {filename}'
    shutil.copy2(path, RUN_DIR / 'data' / filename)

def minimal_clean(text):
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', text)).strip()
def normalize_key(text):
    return minimal_clean(text).casefold()
source_train = pd.read_parquet(next(DATA_DIR.glob('train-*.parquet')))
source_test = pd.read_parquet(next(DATA_DIR.glob('test-*.parquet')))
assert len(source_train) == 1000 and len(source_test) == 320
for frame, prefix in [(source_train, 'train'), (source_test, 'test')]:
    assert frame[['text', 'label']].notna().all().all()
    assert frame['label'].isin(LABELS).all()
    assert frame['text'].map(lambda x:isinstance(x, str) and bool(x.strip())).all()
    frame['row_id'] = [f'{prefix}:{i:04d}' for i in range(len(frame))]
    frame['clean_text'] = frame['text'].map(minimal_clean)
    frame['text_key'] = frame['text'].map(normalize_key)
    frame['word_count'] = frame['clean_text'].str.split().str.len()
    assert frame.groupby('text_key')['label'].nunique().max() == 1, 'Conflicting duplicate labels: review before training.'
assert not set(source_train.text_key) & set(source_test.text_key), 'Exact train/test leakage: review protocol.'
duplicate_count = int(source_train.duplicated('text_key').sum())
development = source_train.drop_duplicates('text_key').copy()
train_df, val_df = train_test_split(development, test_size=CONFIG['validation_fraction'],
                                  stratify=development.label, random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
test_df = source_test.reset_index(drop=True).copy()
assert not set(train_df.text_key) & set(val_df.text_key)
assert not set(train_df.row_id) & set(val_df.row_id)
SPLITS = {'train':train_df, 'validation':val_df, 'test':test_df}
split_manifest = pd.concat([f[['row_id','label','text_key']].assign(split=s) for s,f in SPLITS.items()])
split_manifest.to_csv(RUN_DIR / 'split_manifest.csv', index=False)
quality = {'missing_values':0, 'conflicting_duplicates':0, 'train_duplicates_removed':duplicate_count,
           'cross_split_exact_duplicates':0, 'split_sizes':{s:len(f) for s,f in SPLITS.items()}}
save_json('data_quality.json', quality)
display(pd.DataFrame([{'split':s, 'rows':len(f), **{NAMES[c]:int((f.label==c).sum()) for c in LABELS}}
                      for s,f in SPLITS.items()]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
counts = pd.DataFrame({s:f.label.value_counts().reindex(LABELS, fill_value=0).to_numpy()
                       for s,f in SPLITS.items()}, index=NAMES)
counts.plot.bar(ax=axes[0], rot=0, color=['#1b6a96','#dd8a33','#697582'])
axes[0].set(title='Label counts by split', ylabel='Paragraphs', xlabel='Source label')
for s,f in SPLITS.items():
    axes[1].hist(f.word_count, bins=np.arange(0, 701, 25), alpha=.45, label=f'{s} (n={len(f)})')
axes[1].set(title='Paragraph lengths', xlabel='Whitespace-separated words', ylabel='Paragraphs')
axes[1].legend()
figure('data_profile')
imbalance = train_df.label.value_counts()
print(f'Train largest/smallest class ratio: {imbalance.max()/imbalance.min():.2f}')
display(train_df.sample(5, random_state=SEED)[['row_id','label','word_count','text']])

### 3.1 Linguistic profiling (optional; separate from model features)
Minimal cleaning preserves numbers, negation, punctuation and casing. spaCy's general-English POS tags can describe formal language but do not validate climate semantics. The original submission's linguistic profiling is preserved in the archive. Set `NLP_RUN_POS=true` and install `requirements-pos.txt` to regenerate token-level evidence on **training rows only**. It has no effect on model selection and is excluded from the lightweight run.

In [ ]:
if CONFIG['run_pos']:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    tokens = [{'row_id':rid,'token':t.text,'lemma':t.lemma_,'pos':t.pos_}
              for rid,doc in zip(train_df.row_id, nlp.pipe(train_df.clean_text)) for t in doc]
    tokens_df = pd.DataFrame(tokens)
    save_table('pos_tokens', tokens_df)
    tokens_df.pos.value_counts().plot.bar(color='#1b6a96')
    plt.title('Training-corpus POS profile — general-English tagger')
    plt.ylabel('Tokens')
    figure('pos_profile')
    display(tokens_df.head(10))
else:
    print('Optional POS profiling skipped; enable with NLP_RUN_POS=true.')

## 4. Preprocessing and model alternatives — validation only
We test both **preserving function words** and the original **English stop-word removal**, holding all other TF-IDF+LR settings fixed. Stop-word lists can remove negation; the cell prints exactly which negation words the installed list removes. We compare vocabulary size, retained feature occurrences, validation macro F1 and paired prediction changes. This tests a plausible alternative without claiming that a score difference isolates negation alone.

A full 2×2×2 comparison covers BoW/TF-IDF, logistic regression/Multinomial Naive Bayes, and unigrams/unigrams+bigrams, with both preprocessing options. LR's `C` is chosen within the same validation split. NB uses fixed α=1. All runs use the same training rows, a 1,000-feature cap, and min_df=5. The cap controls cost but can displace rare jargon when bigrams are added. Selection across many candidates can overfit the 200-row validation set; the test score and uncertainty assess the frozen winner.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print('Negation words removed by the English list:', sorted({'not','no','nor','never'} & ENGLISH_STOP_WORDS))
B = CONFIG['baseline']
models, validation_predictions, experiment_rows = {}, {}, []
for representation, vectorizer in [('BoW',CountVectorizer),('TF-IDF',TfidfVectorizer)]:
    for ngram in [(1,1),(1,2)]:
        for stop_words in [None, 'english']:
            for classifier in ['LR','NB']:
                for C in (B['C_grid'] if classifier=='LR' else [None]):
                    name = f'{representation}|{classifier}|{ngram[1]}gram|stop={stop_words}|C={C}'
                    clf = (LogisticRegression(C=C, max_iter=B['max_iter'], random_state=SEED)
                           if classifier=='LR' else MultinomialNB(alpha=B['alpha']))
                    pipeline = Pipeline([('vectorizer',vectorizer(stop_words=stop_words,
                          ngram_range=ngram, min_df=B['min_df'], max_features=B['max_features'])),('classifier',clf)])
                    start = time.perf_counter()
                    pipeline.fit(train_df.clean_text, train_df.label)
                    fit_seconds = time.perf_counter()-start
                    prediction = pipeline.predict(val_df.clean_text)
                    models[name] = pipeline
                    validation_predictions[name] = prediction
                    experiment_rows.append({'model':name,'representation':representation,'classifier':classifier,
                        'ngram_max':ngram[1], 'stop_words':str(stop_words),'C':C,
                        'validation_macro_f1':f1_score(val_df.label,prediction,labels=LABELS,average='macro',zero_division=0),
                        'validation_risk_recall':recall_score(val_df.label,prediction,labels=[0],average='macro',zero_division=0),
                        'fit_seconds':fit_seconds,'vocabulary_size':len(pipeline[0].vocabulary_)})
experiments = pd.DataFrame(experiment_rows).sort_values(['validation_macro_f1','model'],ascending=[False,True]).reset_index(drop=True)
save_table('validation_experiments', experiments)
SELECTED_NAME = experiments.iloc[0]['model']
selected_model = models[SELECTED_NAME]
selected_val_probs = selected_model.predict_proba(val_df.clean_text)
assert selected_model.classes_.tolist() == LABELS
joblib.dump(selected_model, RUN_DIR/'models/classical.joblib')
save_json('selected_classical.json', experiments.iloc[0].to_dict())
print('Frozen classical choice:', SELECTED_NAME)
display(experiments.head(8).round(4))

In [ ]:
preprocessing_rows=[]
for stops in [None, 'english']:
    name=f'TF-IDF|LR|1gram|stop={stops}|C=1.0'
    p=models[name]
    analyzer=p[0].build_analyzer()
    vocab=p[0].vocabulary_
    occurrences=[sum(t in vocab for t in analyzer(x)) for x in train_df.clean_text]
    preprocessing_rows.append({'preprocessing':f'stop_words={stops}',
       'vocabulary_size':len(vocab),'mean_retained_feature_occurrences':np.mean(occurrences),
       'validation_macro_f1':f1_score(val_df.label, validation_predictions[name],labels=LABELS,average='macro',zero_division=0)})
preprocessing=pd.DataFrame(preprocessing_rows)
base=preprocessing.iloc[0].validation_macro_f1
preprocessing['delta_f1_vs_preserve']=preprocessing.validation_macro_f1-base
save_table('preprocessing_comparison',preprocessing)
display(preprocessing.round(4))
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
axes[0].bar(preprocessing.preprocessing,preprocessing.mean_retained_feature_occurrences,color=['#1b6a96','#dd8a33'])
axes[0].set(ylabel='Mean feature occurrences / train paragraph',title='How much lexical evidence is retained?')
axes[1].bar(preprocessing.preprocessing,preprocessing.validation_macro_f1,color=['#1b6a96','#dd8a33'])
axes[1].set(ylim=(0,1),ylabel='Validation macro F1',title='Controlled preprocessing comparison (n=200)')
for i,v in enumerate(preprocessing.validation_macro_f1):axes[1].text(i,v+.015,f'{v:.3f}',ha='center')
figure('preprocessing_comparison')
changed = validation_predictions['TF-IDF|LR|1gram|stop=None|C=1.0'] != validation_predictions['TF-IDF|LR|1gram|stop=english|C=1.0']
print(f'Preprocessing changes {changed.sum()}/{len(val_df)} validation predictions.')
display(val_df.loc[changed,['row_id','label','text']].head(3))

In [ ]:
# Best C within each otherwise fixed comparison; every score is validation, not test.
ablation=experiments.sort_values('validation_macro_f1',ascending=False).drop_duplicates(
    ['representation','classifier','ngram_max','stop_words']).copy()
reference=float(experiments.loc[experiments.model=='TF-IDF|LR|1gram|stop=None|C=1.0','validation_macro_f1'].iloc[0])
ablation['delta_vs_reference']=ablation.validation_macro_f1-reference
save_table('validation_ablation',ablation)
ordered=ablation.sort_values('validation_macro_f1')
fig,ax=plt.subplots(figsize=(11,7))
labels=[f'{r.representation} / {r.classifier} / 1–{r.ngram_max} grams / stop={r.stop_words}' for r in ordered.itertuples()]
ax.barh(labels,ordered.validation_macro_f1,color='#1b6a96')
ax.set(xlim=(0,1),xlabel='Validation macro F1',title='Controlled model alternatives — same 800 training / 200 validation rows')
for i,v in enumerate(ordered.validation_macro_f1):ax.text(v+.01,i,f'{v:.3f}',va='center',fontsize=9)
figure('validation_ablation')
# Pure pairs differ only in the named factor; positive delta favors the second value.
pair_rows=[]
for factor,lo,hi in [('representation','BoW','TF-IDF'),('classifier','NB','LR'),('ngram_max',1,2),('stop_words','None','english')]:
    controls=[c for c in ['representation','classifier','ngram_max','stop_words'] if c!=factor]
    for keys,g in ablation.groupby(controls):
        lookup=g.set_index(factor).validation_macro_f1
        if lo in lookup and hi in lookup:
            pair_rows.append({'factor':factor,'controls':str(keys),'comparison':f'{hi} minus {lo}',
                              'delta_validation_f1':float(lookup[hi]-lookup[lo])})
paired_ablations=pd.DataFrame(pair_rows)
save_table('paired_ablation_deltas',paired_ablations)
display(paired_ablations.groupby('factor').delta_validation_f1.agg(['min','median','max']).round(4))
print('Effects depend on the controls; no blanket claim that LR, TF-IDF or bigrams always wins.')

### 4.1 Training vocabulary and overlap
Vocabulary overlap is descriptive and is fitted on training text only. Shared climate terminology may be useful for the domain yet insufficient to distinguish labels. The top-term plot shows class-average TF-IDF, not causal word importance.

In [ ]:
descriptive_vectorizer=TfidfVectorizer(min_df=5,max_features=1000,stop_words=None)
X_train_descriptive=descriptive_vectorizer.fit_transform(train_df.clean_text)
terms=descriptive_vectorizer.get_feature_names_out()
top_sets=[]
fig,axes=plt.subplots(1,3,figsize=(14,4.8))
for label,ax in zip(LABELS,axes):
    mean=np.asarray(X_train_descriptive[train_df.label.to_numpy()==label].mean(axis=0)).ravel()
    indices=mean.argsort()[-10:]
    ax.barh(terms[indices],mean[indices],color=COLORS[label])
    ax.set(title=f'{NAMES[label]} — training terms',xlabel='Mean TF-IDF / paragraph')
    top_sets.append(set(terms[mean.argsort()[-100:]]))
figure('top_training_terms')
overlap=np.array([[len(a&b)/len(a|b) for b in top_sets] for a in top_sets])
fig,ax=plt.subplots(figsize=(5.5,4.7))
image=ax.imshow(overlap,vmin=0,vmax=1,cmap='Blues')
ax.set(xticks=LABELS,yticks=LABELS,xticklabels=NAMES,yticklabels=NAMES,title='Top-100 vocabulary Jaccard overlap')
for i in LABELS:
    for j in LABELS:ax.text(j,i,f'{overlap[i,j]:.2f}',ha='center',va='center',color='white' if overlap[i,j]>.5 else '#202830')
fig.colorbar(image,ax=ax,label='Intersection / union')
figure('vocabulary_overlap')

## 5. Freeze routing before test evaluation
The original notebook imposed 0.75/0.90 probability thresholds and an alleged short-text performance gap without a supporting slice test. Here a single threshold is chosen using validation only, with explicit retained accuracy, coverage and support requirements. If no threshold qualifies, **every paragraph goes to review**. This implements a fail-closed policy; it does not establish calibration or fairness. Length-aware thresholds would need enough validation examples per length slice and a new confirmation set.

In [ ]:
def routing_policy(y, probs):
    pred=probs.argmax(axis=1); confidence=probs.max(axis=1)
    rows=[]
    for threshold in CONFIG['routing_threshold_grid']:
        accepted=confidence>=threshold
        rows.append({'threshold':threshold,'retained_n':int(accepted.sum()),'coverage':float(accepted.mean()),
                     'retained_accuracy':float(np.mean(pred[accepted]==np.asarray(y)[accepted])) if accepted.any() else np.nan})
    table=pd.DataFrame(rows)
    D=CONFIG['decision']
    eligible=table[(table.retained_n>=D['min_validation_retained_n']) &
                   (table.coverage>=D['min_coverage']) & (table.retained_accuracy>=D['min_retained_accuracy'])]
    threshold=float(eligible.sort_values(['coverage','threshold'],ascending=[False,True]).iloc[0].threshold) if len(eligible) else None
    return threshold,table
classical_threshold,classical_routing_validation=routing_policy(val_df.label,selected_val_probs)
save_table('classical_routing_validation',classical_routing_validation)
save_json('classical_routing_policy.json',{'threshold':classical_threshold,'source':'validation only','if_null':'review all'})
display(classical_routing_validation.round(3))
print('Frozen routing threshold:',classical_threshold,'(None means review all)')

## 6. Fine-tune DistilBERT — GPU profile
DistilBERT is a smaller contextual alternative to full BERT. We use a pinned base checkpoint, a fresh three-class head, learning rate 2e-5, batch size 16, FP32, weight decay 0.01, and at most three epochs. Dynamic padding reduces wasted work. Early stopping selects the best validation macro F1 and stops after one epoch without improvement. Seeds are set **before head initialization**. [Model documentation](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/distilbert), [Trainer reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer).

**Constraint trade-off:** max_length=128 limits memory and runtime but may discard a paragraph's conclusion. The code measures actual subword truncation and saves its slice performance. The original CPU training took **1,886.7 seconds** for three epochs; validation F1 was **0.7951 → 0.8295 → 0.8295**, while validation loss continued falling (**0.5990 → 0.4696 → 0.4517**). This supports a plateau/compute trade-off, not the original claim of observed loss divergence. The revised code stops after the plateau and saves actual training logs. A 256-token or chunked model is an untested improvement to evaluate if truncation errors are concentrated.

Full runs train seeds 42/43/44 on the same rows; seed 42 is the prespecified primary comparison, not the best test seed. Secondary seeds quantify initialization/order variation. CPU runs keep transformer fields **missing**, never zero, and do not imply a successful fine-tuning validation.

In [ ]:
HAS_CUDA=False
if MODE!='baseline':
    try:
        import torch
        HAS_CUDA=torch.cuda.is_available()
    except ImportError:
        pass
if MODE=='full' and not HAS_CUDA:
    raise RuntimeError('Full profile requires CUDA. Select a Colab GPU kernel, install GPU requirements, restart, then Run all.')
RUN_TRANSFORMER=MODE!='baseline' and HAS_CUDA
transformer_runs=[]
transformer_status='pending: CUDA run not performed'
if RUN_TRANSFORMER:
    os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
    os.environ['USE_TF']='0'
    from datasets import Dataset, DatasetDict
    from transformers import (AutoTokenizer,AutoModelForSequenceClassification,DataCollatorWithPadding,
                              Trainer,TrainingArguments,EarlyStoppingCallback,set_seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False
    tokenizer=AutoTokenizer.from_pretrained(CONFIG['model_id'],revision=CONFIG['model_revision'])
    T=CONFIG['transformer']
    token_lengths={s:np.array([len(ids) for ids in tokenizer(f.clean_text.tolist(),truncation=False)['input_ids']]) for s,f in SPLITS.items()}
    truncation=pd.DataFrame([{'split':s,'n':len(lengths),'truncated_n':int((lengths>T['max_length']).sum()),
                              'truncated_fraction':float((lengths>T['max_length']).mean()),'p95_subwords':float(np.quantile(lengths,.95))}
                             for s,lengths in token_lengths.items()])
    save_table('subword_truncation',truncation)
    display(truncation.round(3))
    # Only input_ids and attention_mask are produced by the pinned DistilBERT tokenizer.
    tokenized=DatasetDict({s:Dataset.from_dict({'text':f.clean_text.tolist(),'label':f.label.tolist()}) for s,f in SPLITS.items()})
    tokenized=tokenized.map(lambda batch:tokenizer(batch['text'],truncation=True,max_length=T['max_length']),
                            batched=True,remove_columns=['text'])
    def hf_metrics(output):
        pred=output.predictions.argmax(axis=1)
        return {'macro_f1':f1_score(output.label_ids,pred,labels=LABELS,average='macro',zero_division=0),
                'accuracy':accuracy_score(output.label_ids,pred)}
    ENVIRONMENT.update({'torch':torch.__version__,'cuda_version':torch.version.cuda,
                        'gpu':torch.cuda.get_device_name(0),'deterministic_algorithms':True,'fp16':False})
    save_json('environment.json',ENVIRONMENT)
else:
    print(transformer_status+'. Classical analysis will complete; transformer-dependent gates remain pending.')

In [ ]:
if RUN_TRANSFORMER:
    for run_seed in CONFIG['transformer']['seeds']:
        set_seed(run_seed)
        model=AutoModelForSequenceClassification.from_pretrained(CONFIG['model_id'],revision=CONFIG['model_revision'],
                num_labels=3,id2label=dict(enumerate(NAMES)),label2id={name:i for i,name in enumerate(NAMES)},
                attn_implementation='eager')
        checkpoint_dir=RUN_DIR/'models'/f'distilbert_seed_{run_seed}'
        args=TrainingArguments(output_dir=str(checkpoint_dir/'checkpoints'),num_train_epochs=T['epochs'],
             learning_rate=T['learning_rate'],per_device_train_batch_size=T['batch_size'],
             per_device_eval_batch_size=T['batch_size'],weight_decay=T['weight_decay'],warmup_ratio=T['warmup_ratio'],
             eval_strategy='epoch',save_strategy='epoch',logging_strategy='steps',logging_steps=10,
             load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,
             save_total_limit=1,seed=run_seed,data_seed=run_seed,full_determinism=True,fp16=False,bf16=False,
             optim='adamw_torch',report_to='none',dataloader_num_workers=0,dataloader_pin_memory=True)
        trainer=Trainer(model=model,args=args,train_dataset=tokenized['train'],eval_dataset=tokenized['validation'],
              processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=hf_metrics,
              callbacks=[EarlyStoppingCallback(early_stopping_patience=T['early_stopping_patience'])])
        torch.cuda.reset_peak_memory_stats()
        start=time.perf_counter()
        trainer.train()
        torch.cuda.synchronize()
        train_seconds=time.perf_counter()-start
        trainer.save_model(str(checkpoint_dir/'best'))
        tokenizer.save_pretrained(checkpoint_dir/'best')
        trainer.state.save_to_json(str(checkpoint_dir/'trainer_state.json'))
        save_json(f'models/distilbert_seed_{run_seed}/training_arguments.json',args.to_dict())
        val_logits=trainer.predict(tokenized['validation']).predictions
        val_probs=torch.softmax(torch.tensor(val_logits),dim=-1).numpy()
        threshold,routing_table=routing_policy(val_df.label,val_probs)
        save_table(f'transformer_seed_{run_seed}_routing_validation',routing_table)
        # Warm-up is excluded from timed prediction. Test labels are not used for selection.
        trainer.predict(tokenized['test'].select(range(min(16,len(test_df)))))
        torch.cuda.synchronize()
        start=time.perf_counter()
        output=trainer.predict(tokenized['test'])
        torch.cuda.synchronize()
        inference_seconds=time.perf_counter()-start
        test_probs=torch.softmax(torch.tensor(output.predictions),dim=-1).numpy()
        transformer_runs.append({'seed':run_seed,'probs':test_probs,'val_probs':val_probs,'threshold':threshold,
            'fit_seconds':train_seconds,'inference_ms_per_paragraph':1000*inference_seconds/len(test_df),
            'artifact_mb':sum(p.stat().st_size for p in (checkpoint_dir/'best').rglob('*') if p.is_file())/1e6,
            'peak_gpu_mb':torch.cuda.max_memory_allocated()/1e6,'history':trainer.state.log_history})
        np.save(checkpoint_dir/'test_probabilities.npy',test_probs)
        np.save(checkpoint_dir/'validation_probabilities.npy',val_probs)
        save_json(f'models/distilbert_seed_{run_seed}/routing_policy.json',{'threshold':threshold,'source':'validation'})
        del trainer,model
        torch.cuda.empty_cache()
    transformer_status='complete: controlled GPU training and predictions saved'
    print(transformer_status)

In [ ]:
if RUN_TRANSFORMER:
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    for result in transformer_runs:
        evaluation=[r for r in result['history'] if 'eval_macro_f1' in r]
        axes[0].plot([r['epoch'] for r in evaluation],[r['eval_loss'] for r in evaluation],marker='o',label=f"seed {result['seed']}")
        axes[1].plot([r['epoch'] for r in evaluation],[r['eval_macro_f1'] for r in evaluation],marker='o',label=f"seed {result['seed']}")
    axes[0].set(title='Recorded validation loss',xlabel='Epoch',ylabel='Cross-entropy loss')
    axes[1].set(title='Recorded validation macro F1',xlabel='Epoch',ylabel='Macro F1',ylim=(0,1))
    for ax in axes:ax.legend()
    figure('controlled_training_curves')
else:
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].plot([1,2,3],LEGACY['training']['validation_loss'],'o-',color='#1b6a96')
    axes[1].plot([1,2,3],LEGACY['training']['validation_macro_f1'],'o-',color='#167c70')
    axes[0].set(title='Historical validation loss — archived output',xlabel='Epoch',ylabel='Loss')
    axes[1].set(title='Historical validation F1 — archived output',xlabel='Epoch',ylabel='Macro F1',ylim=(0,1))
    for ax in axes:ax.set_xticks([1,2,3])
    figure('historical_training_curves')
    print('Historical curves are not the revised controlled experiment. Loss falls; F1 plateaus after epoch 2.')

## 7. Locked-model test evaluation and uncertainty
Only the selected classical candidate and a majority-class sanity baseline are evaluated here; unselected candidates remain validation-only. Transformer seed 42 uses the same test row order when present. Stratified paired bootstrap resamples the same test indices for both models, preserving the observed class counts. Its 95% interval is conditional on this sample and fitted models; it cannot establish issuer-level independence or deployment readiness.

In [ ]:
def metrics(y,probs):
    pred=probs.argmax(axis=1)
    return {'accuracy':accuracy_score(y,pred),
            'macro_f1':f1_score(y,pred,labels=LABELS,average='macro',zero_division=0),
            'micro_f1':f1_score(y,pred,labels=LABELS,average='micro',zero_division=0),
            'macro_average_precision':average_precision_score(label_binarize(y,classes=LABELS),probs,average='macro'),
            'risk_recall':recall_score(y,pred,labels=[0],average='macro',zero_division=0)}
selected_model.predict_proba(test_df.clean_text.iloc[:16])  # warm-up
start=time.perf_counter()
classical_probs=selected_model.predict_proba(test_df.clean_text)
classical_ms=1000*(time.perf_counter()-start)/len(test_df)
majority=DummyClassifier(strategy='most_frequent').fit(train_df.clean_text,train_df.label)
majority_probs=majority.predict_proba(test_df.clean_text)
MODEL_PROBS={'Majority baseline':majority_probs,'Selected classical':classical_probs}
for result in transformer_runs:MODEL_PROBS[f"DistilBERT seed {result['seed']}"]=result['probs']
y_true=test_df.label.to_numpy()
rng=np.random.default_rng(SEED)
bootstrap_indices=np.array([np.concatenate([rng.choice(np.flatnonzero(y_true==c),size=int((y_true==c).sum()),replace=True)
                                           for c in LABELS]) for _ in range(CONFIG['bootstrap_resamples'])])
def bootstrap_f1(pred):
    # Confusion-matrix arithmetic avoids repeatedly fitting or invoking model predictions.
    draws=[]
    for idx in bootstrap_indices:
        cm=np.bincount(3*y_true[idx]+pred[idx],minlength=9).reshape(3,3)
        denominator=cm.sum(0)+cm.sum(1)
        draws.append(np.divide(2*cm.diagonal(),denominator,out=np.zeros(3,dtype=float),where=denominator!=0).mean())
    return np.array(draws)
metric_rows=[]; bootstrap_scores={}
for name,probs in MODEL_PROBS.items():
    assert probs.shape==(len(test_df),3) and np.isfinite(probs).all()
    assert np.allclose(probs.sum(1),1,atol=1e-5)
    stats=metrics(y_true,probs)
    bootstrap_scores[name]=bootstrap_f1(probs.argmax(1))
    low,high=np.quantile(bootstrap_scores[name],[.025,.975])
    metric_rows.append({'model':name,**stats,'macro_f1_ci_low':low,'macro_f1_ci_high':high,'test_n':len(test_df)})
    predictions=test_df[['row_id','text','label','word_count']].copy()
    predictions['prediction']=probs.argmax(1)
    predictions['confidence']=probs.max(1)
    for c in LABELS:predictions[f'p_{NAMES[c]}']=probs[:,c]
    predictions.to_csv(RUN_DIR/'predictions'/(re.sub(r'[^a-z0-9]+','_',name.lower())+'.csv'),index=False)
    save_table(re.sub(r'[^a-z0-9]+','_',name.lower())+'_class_metrics',
               pd.DataFrame(classification_report(y_true,probs.argmax(1),labels=LABELS,target_names=NAMES,output_dict=True,zero_division=0)).T.reset_index(names='class'))
metric_table=pd.DataFrame(metric_rows)
save_table('test_metrics',metric_table)
display(metric_table.round(4))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
y=np.arange(len(metric_table))
axes[0].errorbar(metric_table.macro_f1,y,
     xerr=[metric_table.macro_f1-metric_table.macro_f1_ci_low,metric_table.macro_f1_ci_high-metric_table.macro_f1],
     fmt='o',capsize=4,color='#1b6a96')
axes[0].set(yticks=y,yticklabels=metric_table.model,xlim=(0,1),xlabel='Macro F1; 95% stratified bootstrap interval',
            title=f'Official test comparison (n={len(test_df)})')
axes[0].axvline(CONFIG['decision']['min_macro_f1'],ls='--',color='#8b572a',label='Pilot target')
axes[0].legend(loc='lower right')
axes[1].bar(metric_table.model,metric_table.risk_recall,color='#b54736')
axes[1].axhline(CONFIG['decision']['min_risk_recall'],ls='--',color='#8b572a',label='Risk recall target')
axes[1].set(ylim=(0,1.04),ylabel='Recall of labelled risk paragraphs',title='How many risk paragraphs are found?')
axes[1].tick_params(axis='x',rotation=30)
axes[1].legend()
figure('test_comparison')
paired_result=None
if transformer_runs:
    delta_samples=bootstrap_scores['DistilBERT seed 42']-bootstrap_scores['Selected classical']
    delta=float(metric_table.set_index('model').loc['DistilBERT seed 42','macro_f1']-metric_table.set_index('model').loc['Selected classical','macro_f1'])
    low,high=np.quantile(delta_samples,[.025,.975])
    paired_result={'primary_seed':42,'delta_macro_f1':delta,'ci_low':float(low),'ci_high':float(high),
                   'method':'paired class-stratified percentile bootstrap','resamples':CONFIG['bootstrap_resamples']}
    save_json('paired_comparison.json',paired_result)
    print(f'Primary transformer minus classical: {delta:+.4f} F1; 95% paired CI [{low:+.4f}, {high:+.4f}]')
    secondary=metric_table[metric_table.model.str.startswith('DistilBERT')]
    print(f'Prespecified-seed test F1 mean={secondary.macro_f1.mean():.4f}, SD={secondary.macro_f1.std(ddof=1):.4f}; 3 seeds is limited evidence.')
else:
    print('Controlled transformer comparison pending. Historical 0.8470 is not mixed into this plot.')

### 7.1 Resource trade-offs and test routing
Inference is warmed-up batch wall time per paragraph, including the model wrapper; it is **not** single-request p95 latency. CPU and GPU timings describe separate deployment profiles and do not isolate architecture speed. Measure the intended service on its own hardware before adopting it. Fit time excludes classical candidate search; search time is also reported.

In [ ]:
resource_rows=[{'model':'Selected classical','device':'CPU','fit_seconds':float(experiments.iloc[0].fit_seconds),
               'search_fit_seconds':float(experiments.fit_seconds.sum()),'inference_ms_per_paragraph':classical_ms,
               'artifact_mb':(RUN_DIR/'models/classical.joblib').stat().st_size/1e6}]
for r in transformer_runs:
    resource_rows.append({'model':f"DistilBERT seed {r['seed']}",'device':ENVIRONMENT.get('gpu','CUDA'),
            **{k:r[k] for k in ['fit_seconds','inference_ms_per_paragraph','artifact_mb','peak_gpu_mb']}})
resources=pd.DataFrame(resource_rows)
save_table('resource_comparison',resources)
display(resources.round(3))
policies={'Selected classical':classical_threshold}
policies.update({f"DistilBERT seed {r['seed']}":r['threshold'] for r in transformer_runs})
routing_rows=[]
for name,threshold in policies.items():
    probs=MODEL_PROBS[name]
    accepted=probs.max(1)>=threshold if threshold is not None else np.zeros(len(test_df),dtype=bool)
    routing_rows.append({'model':name,'threshold':threshold,'retained_n':int(accepted.sum()),
        'coverage':float(accepted.mean()),'review_fraction':float(1-accepted.mean()),
        'retained_accuracy':float((probs.argmax(1)[accepted]==y_true[accepted]).mean()) if accepted.any() else np.nan,
        'missed_risk_in_retained':int(((y_true==0)&(probs.argmax(1)!=0)&accepted).sum())})
routing_test=pd.DataFrame(routing_rows)
save_table('routing_test',routing_test)
display(routing_test.round(3))
fig,ax=plt.subplots(figsize=(10,4))
x=np.arange(len(routing_test))
ax.bar(x,routing_test.coverage,label='Retained by frozen policy',color='#167c70')
ax.bar(x,routing_test.review_fraction,bottom=routing_test.coverage,label='Human review',color='#dd8a33')
ax.set(xticks=x,xticklabels=routing_test.model,ylim=(0,1),ylabel='Share of test paragraphs',title='Review workload — thresholds selected on validation only')
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),ncol=2)
figure('review_workload')

## 8. Documented failures and uncertainty
These diagnostics describe test performance; they do not authorize retuning against this test set. Rule-based slices are **proxies**, overlap, and are not demographic fairness groups. Long text means above the **training** 75th percentile. Short text means <15 words, regardless of whether enough such rows exist. The jargon, forward-looking and risk+mitigation patterns are fixed here for investigation; a match does not prove the paragraph's intended sentiment.

In [ ]:
nontrivial=[n for n in MODEL_PROBS if n!='Majority baseline']
fig,axes=plt.subplots(1,len(nontrivial),figsize=(5*len(nontrivial),4.5),squeeze=False)
for name,ax in zip(nontrivial,axes.ravel()):
    cm=confusion_matrix(y_true,MODEL_PROBS[name].argmax(1),labels=LABELS)
    im=ax.imshow(cm,cmap='Blues',vmin=0,vmax=max((y_true==c).sum() for c in LABELS))
    ax.set(xticks=LABELS,yticks=LABELS,xticklabels=NAMES,yticklabels=NAMES,xlabel='Predicted',ylabel='Source label',title=name)
    for i in LABELS:
        for j in LABELS:ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>80 else '#202830')
figure('confusion_matrices')
long_cutoff=float(train_df.word_count.quantile(.75))
slice_masks={
 'All paragraphs':np.ones(len(test_df),bool),
 'Short (<15 words)':test_df.word_count.to_numpy()<15,
 'Long (>train Q75)':test_df.word_count.to_numpy()>long_cutoff,
 'Jargon proxy':test_df.clean_text.str.contains(r'\b(?:TCFD|GHG|PRA|FCA)\b|Scope\s+[123]',case=False,regex=True).to_numpy(),
 'Forward-looking proxy':test_df.clean_text.str.contains(r'\b(?:aim|goal|target|plan|commit)\w*\b',case=False).to_numpy(),
 'Risk + mitigation proxy':(test_df.clean_text.str.contains(r'\brisk\w*\b',case=False)&test_df.clean_text.str.contains(r'\b(?:mitigat|reduc|manag)\w*\b',case=False)).to_numpy()
}
if RUN_TRANSFORMER:slice_masks['Truncated at 128 subwords']=token_lengths['test']>CONFIG['transformer']['max_length']
slice_rows=[]
for name in nontrivial:
    pred=MODEL_PROBS[name].argmax(1)
    for slice_name,mask in slice_masks.items():
        n=int(mask.sum())
        slice_rows.append({'model':name,'slice':slice_name,'n':n,'risk_n':int(((y_true==0)&mask).sum()),
          'errors':int(((pred!=y_true)&mask).sum()),'accuracy':accuracy_score(y_true[mask],pred[mask]) if n else np.nan,
          'macro_f1':f1_score(y_true[mask],pred[mask],labels=LABELS,average='macro',zero_division=0) if n else np.nan,
          'risk_recall':float((pred[mask & (y_true==0)]==0).mean()) if (mask & (y_true==0)).any() else np.nan})
slices=pd.DataFrame(slice_rows)
save_table('error_slices',slices)
display(slices.round(3))
fig,ax=plt.subplots(figsize=(11,5))
for j,name in enumerate(nontrivial):
    d=slices[(slices.model==name)&(slices.n>0)]
    ax.plot(d.slice,d.accuracy,marker='o',label=name)
ax.set(ylim=(0,1.05),ylabel='Accuracy (overlapping slices)',title=f'Failure probes — long cutoff {long_cutoff:g} training words')
ax.tick_params(axis='x',rotation=25)
ax.legend()
figure('slice_comparison')
print('Three-class macro F1 uses all 3 labels even if a small slice lacks a class; inspect support before comparing slices.')

In [ ]:
# Audit a residual near-duplicate risk without fitting vocabulary on test.
from sklearn.metrics.pairwise import cosine_similarity
similarity_vectorizer=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=20000)
train_similarity=similarity_vectorizer.fit_transform(train_df.clean_text)
test_similarity=similarity_vectorizer.transform(test_df.clean_text)
similarities=cosine_similarity(test_similarity,train_similarity)
nearest=similarities.argmax(1)
near=pd.DataFrame({'test_row_id':test_df.row_id,'train_row_id':train_df.row_id.iloc[nearest].to_numpy(),
                   'cosine_similarity':similarities.max(1)})
save_table('nearest_train_neighbors',near)
print(f'Test paragraphs with character TF-IDF similarity ≥0.90 to training: {(near.cosine_similarity>=.90).sum()}/{len(test_df)}')
display(near.sort_values('cosine_similarity',ascending=False).head(5).round(3))
errors=test_df[['row_id','text','label','word_count']].copy()
errors['prediction']=classical_probs.argmax(1)
errors['confidence']=classical_probs.max(1)
errors['failure_pair']=[f'{NAMES[a]} → {NAMES[b]}' for a,b in zip(errors.label,errors.prediction)]
for name,mask in slice_masks.items():errors[name]=mask
errors=errors[errors.label!=errors.prediction].sort_values(['confidence','row_id'],ascending=[False,True])
save_table('classical_errors',errors)
# Up to 20 deterministic, highest-confidence disagreements; not a random error sample.
display(errors[['row_id','label','prediction','confidence','word_count','text']].head(20).round(3))
error_pair=errors.failure_pair.value_counts()
if len(error_pair):
    print('Most frequent documented confusion:',error_pair.index[0],f'(n={error_pair.iloc[0]})')

## 9. Explainability tied to an actual failure
We replace the unverified SHAP narrative with **exact local linear contributions** for a training-fitted logistic regression, and a **word-deletion sensitivity probe** for the selected classifier. The explained paragraph is the highest-confidence classical disagreement from the preceding saved error table. It is a real labelled test row, not a synthetic example assigned a presumed correct label.

For LR, TF-IDF × (predicted-class weight − true-class weight), plus the intercept difference, exactly reconstructs the predicted-vs-true logit gap. If NB wins selection, the reference LR explains its own decision separately; the deletion probe still explains the selected model. Local weights show model associations, not linguistic causes, label correctness, or human reasoning. Word deletion changes context and TF-IDF normalization; it is a perturbation diagnostic, not causal evidence or SHAP.

In [ ]:
explain_findings={}
if len(errors):
    example=errors.iloc[0]
    text=example['text']
    true_class=int(example.label)
    if isinstance(selected_model[-1],LogisticRegression):
        explanation_model=selected_model
        explanation_name=SELECTED_NAME
    else:
        explanation_name='TF-IDF|LR|1gram|stop=None|C=1.0'
        explanation_model=models[explanation_name]
    vectorizer,linear=explanation_model[0],explanation_model[-1]
    predicted_class=int(explanation_model.predict([minimal_clean(text)])[0])
    if predicted_class==true_class:
        # Compare the selected model's mistaken class against the reference LR's true class.
        predicted_class=int(example.prediction)
    x=vectorizer.transform([minimal_clean(text)])
    delta_weights=linear.coef_[predicted_class]-linear.coef_[true_class]
    contributions=x.toarray().ravel()*delta_weights
    active=np.flatnonzero(contributions)
    top=active[np.argsort(np.abs(contributions[active]))[-12:]]
    explanation=pd.DataFrame({'term':vectorizer.get_feature_names_out()[top],
                              'predicted_minus_true_logit_contribution':contributions[top]})
    save_table('local_linear_explanation',explanation)
    logits=linear.decision_function(x)[0]
    reconstructed=float(contributions.sum()+linear.intercept_[predicted_class]-linear.intercept_[true_class])
    assert np.isclose(reconstructed,logits[predicted_class]-logits[true_class])
    fig,ax=plt.subplots(figsize=(10,5))
    ax.barh(explanation.term,explanation.predicted_minus_true_logit_contribution,
            color=['#b54736' if v>0 else '#167c70' for v in explanation.predicted_minus_true_logit_contribution])
    ax.axvline(0,color='#526b80',lw=.8)
    ax.set(title=f"Actual error {example.row_id}: {NAMES[int(example.label)]} → {NAMES[int(example.prediction)]}",
           xlabel=f'Contribution to {NAMES[predicted_class]} minus {NAMES[true_class]} logit (LR)')
    figure('local_error_explanation')
    selected_class=int(example.prediction)
    word_spans=list(re.finditer(r'\b\w+\b',text))
    # Bound compute and preserve repeated-word identities with start/end offsets.
    candidate_spans=word_spans[:80]
    perturbed=[minimal_clean(text[:m.start()]+' '+text[m.end():]) for m in candidate_spans]
    original_probability=float(selected_model.predict_proba([minimal_clean(text)])[0,selected_class])
    deletion_probs=selected_model.predict_proba(perturbed)[:,selected_class]
    deletion=pd.DataFrame({'word':[m.group() for m in candidate_spans],'start':[m.start() for m in candidate_spans],
                           'end':[m.end() for m in candidate_spans],
                           'drop_in_selected_class_probability':original_probability-deletion_probs})
    deletion=deletion.sort_values('drop_in_selected_class_probability',ascending=False)
    save_table('word_deletion_sensitivity',deletion)
    display(Markdown(f"**Explained row `{example.row_id}`:** {text}\n\nSource label: **{NAMES[true_class]}**; selected prediction: **{NAMES[selected_class]}**; model score **{example.confidence:.3f}**. Explainability model: `{explanation_name}`."))
    display(deletion.head(8).round(4))
    top_support=explanation.sort_values('predicted_minus_true_logit_contribution',ascending=False).iloc[0]
    explain_findings={'row_id':example.row_id,'failure_pair':example.failure_pair,'linear_model':explanation_name,
                     'top_supporting_term':str(top_support.term),'logit_contribution':float(top_support.iloc[1]),
                     'selected_deletion_top_word':str(deletion.iloc[0].word),
                     'selected_deletion_drop':float(deletion.iloc[0].drop_in_selected_class_probability)}
    save_json('explainability_finding.json',explain_findings)
    display(Markdown(f"**Observed finding:** for this documented `{example.failure_pair}` disagreement, `{top_support.term}` contributes {top_support.iloc[1]:+.3f} to the LR mistaken-class margin. Deleting `{deletion.iloc[0].word}` changes the selected model's mistaken-class score by {deletion.iloc[0].drop_in_selected_class_probability:+.3f}. This supports inspecting lexical associations and mixed framing in this row; it does not prove the label is wrong or that all errors have the same cause."))
else:
    print('No classical errors in this test run; no error explanation claimed.')

### 9.1 Paired examples when transformer predictions exist
A transformer gain must be supported by both aggregate metrics and row-level corrections. The following table shows corrected and newly introduced errors by stable row ID. This is evidence of changed decisions, not proof that attention or a specific word caused the improvement. Synthetic risk-mitigation sentences from the original notebook had no ground-truth labels and therefore cannot establish that the transformer was correct.

In [ ]:
if transformer_runs:
    trans_pred=MODEL_PROBS['DistilBERT seed 42'].argmax(1)
    base_pred=classical_probs.argmax(1)
    paired_examples=test_df[['row_id','text','label']].copy()
    paired_examples['classical_prediction']=base_pred
    paired_examples['transformer_prediction']=trans_pred
    paired_examples['outcome']=np.select([(base_pred!=y_true)&(trans_pred==y_true),(base_pred==y_true)&(trans_pred!=y_true),
                              (base_pred!=y_true)&(trans_pred!=y_true)],['corrected','new error','both wrong'],default='both correct')
    save_table('paired_error_examples',paired_examples)
    display(paired_examples.groupby('outcome').size().rename('paragraphs').to_frame())
    display(paired_examples[paired_examples.outcome.isin(['corrected','new error'])].groupby('outcome',sort=False).head(3))
else:
    print('Pending paired transformer predictions. No claim that it fixes a particular failure pattern.')

## 10. Results for stakeholders
The following briefing is generated from the current run's saved metrics and errors. A higher score does not establish that company claims are true. A large review queue means the available evidence does not justify automatic acceptance at the proposed quality target.

In [ ]:
base=metric_table.set_index('model').loc['Selected classical']
major=metric_table.set_index('model').loc['Majority baseline']
route=routing_test.set_index('model').loc['Selected classical']
if len(error_pair):
    failure_sentence=f"The most frequent classical disagreement is **{error_pair.index[0]}** ({int(error_pair.iloc[0])} paragraphs)."
else:
    failure_sentence='No classical disagreements were observed in this test sample.'
briefing=f"""The selected classical system gets **{base.accuracy:.1%}** of the **{len(test_df)}** test paragraphs correct. Its balanced score (macro F1) is **{base.macro_f1:.3f}**, compared with **{major.macro_f1:.3f}** for always choosing the most common training label. It finds **{base.risk_recall:.1%}** of labelled risk paragraphs.

{failure_sentence} The saved examples show what needs analyst attention. Agreement with these labels is a useful test, but cannot establish accuracy on new companies or years.

The validation-selected review policy retains **{route.coverage:.1%}** and sends **{route.review_fraction:.1%}** to people. """
if route.retained_n:
    briefing+=f"Accuracy among the {int(route.retained_n)} retained test paragraphs is **{route.retained_accuracy:.1%}**; this small-sample estimate is not a safety guarantee."
else:
    briefing+='No threshold met all validation gates, so the policy correctly requires review for every paragraph.'
if paired_result:
    briefing+=f"\n\nThe prespecified transformer seed changes balanced score by **{paired_result['delta_macro_f1']:+.3f}** (95% paired interval **{paired_result['ci_low']:+.3f} to {paired_result['ci_high']:+.3f}**). Interpret this alongside the corrected/new-error examples and hardware costs."
else:
    briefing+='\n\nThe revised transformer comparison is pending a GPU run. Historical scores do not settle the revised choice.'
display(Markdown(briefing))
(RUN_DIR/'stakeholder_briefing.md').write_text(briefing+'\n')
# Triangulation: quantify a failure, link actual rows, then give a bounded explanation.
if len(errors):
    dominant=error_pair.index[0]
    example_ids=errors.loc[errors.failure_pair==dominant,'row_id'].head(3).tolist()
    inference=f"Observed failure evidence: confusion counts show {int(error_pair.iloc[0])} cases of {dominant}; actual paragraphs {', '.join(example_ids)} are saved in classical_errors.csv. Local evidence from {explain_findings.get('row_id','unavailable')} probes which lexical cues support a disagreement. This triangulates aggregate errors, labelled text and a local sensitivity/weight diagnostic. Mixed climate framing or lexical shortcuts are plausible explanations; ambiguous labels and sampling variation remain alternatives. It does not demonstrate a causal mechanism."
    display(Markdown(inference))
    (RUN_DIR/'triangulated_inference.md').write_text(inference+'\n')

## 11. Decision and guardrails
The final choice applies the proposed gates from Section 1. **Human review remains mandatory during the pilot**, including retained predictions, until a new issuer/temporal holdout and operating targets are approved. Thresholding is an evaluated candidate mitigation; it is not permission to make consequential decisions automatically.

| Named risk / observed limitation | Feasible mitigation | Evaluation / monitoring check | Decision constraint |
|---|---|---|---|
| Missed risk paragraphs, especially the dominant confusion in Section 8 | Re-annotate mixed-framing training examples with two reviewers; add only reviewed examples to training | Risk recall, risk→neutral counts, reviewer disagreement on a new issuer holdout | No unattended exclusion of potential risks; pause if risk recall <0.85 |
| High model score on an actual error (Section 9) | Review routing and local explanation for disputed cases | Validation-fixed coverage/retained accuracy, high-score error rate; target retained accuracy ≥0.95 | Treat scores as ranking signals; if gates fail, review all |
| Truncation / missing conclusions | Test 256 tokens or paragraph chunking on development data; retain tail context | Subword truncation fraction and truncated/non-truncated risk recall | No claim that 128 tokens covers a complete report; change only after a new evaluation |
| Shared issuers, boilerplate, and near duplicates | Deduplicate exact text; commission issuer-disjoint and time-disjoint sets | Hash overlap, nearest-neighbor similarity, performance on new issuers/years | Random-split scores do not justify new-issuer deployment |
| Domain shift, sparse acronyms, future-goal wording | Label new-domain samples; report jargon and forward-looking slices with support | Slice errors, review rate, input length/jargon drift; manually label 100 new paragraphs monthly in a pilot | Restrict to English climate paragraphs; new language/domain needs validation |
| Misuse as a truth / greenwashing / investment assessment | Show original paragraph, label definition, evidence links and human sign-off | Review decisions and audit output usage; incident count | Classification must not determine investments or environmental compliance |
| Dataset use restrictions | Preserve attribution and source license; resolve permitted use before a commercial pilot | Check dataset/model terms for intended use | Academic evidence does not grant commercial use rights |

**Improvement directly tied to a documented failure:** use the most frequent source→prediction pair and its saved row IDs to create a double-reviewed training annotation guide. Evaluate whether the resulting examples improve that class recall and macro F1 on a fresh holdout while preserving the risk-recall gate. If long/truncated failures dominate, compare 128 vs 256 tokens on validation before the next held-out evaluation. Do not assert that augmentation, longer context, or domain pretraining will improve results until tested.

In [ ]:
D=CONFIG['decision']
def passes_quality(row):return row.macro_f1>=D['min_macro_f1'] and row.risk_recall>=D['min_risk_recall']
base_resource=resources.set_index('model').loc['Selected classical']
base_pass=passes_quality(base) and base_resource.inference_ms_per_paragraph<=D['max_inference_ms_per_paragraph']
choice='Selected classical' if base_pass else 'Human-led review; no model clears all pilot gates'
reasons=[]
if paired_result:
    trans=metric_table.set_index('model').loc['DistilBERT seed 42']
    tr=resources.set_index('model').loc['DistilBERT seed 42']
    transformer_pass=(passes_quality(trans) and trans.risk_recall>=base.risk_recall and
        paired_result['delta_macro_f1']>=D['min_transformer_gain'] and paired_result['ci_low']>0 and
        tr.inference_ms_per_paragraph<=D['max_inference_ms_per_paragraph'])
    if transformer_pass:choice='DistilBERT seed 42 (conditional pilot candidate)'
    reasons.append({'transformer_primary_passes':bool(transformer_pass)})
else:
    reasons.append({'transformer_primary_passes':None,'reason':'controlled GPU evidence pending'})
decision={'choice':choice,'classical_passes':bool(base_pass),'gates':D,'details':reasons,
          'use':'analyst assistance with human review',
          'change_choice_if':'Transformer meets paired-gain, risk-recall and latency gates on controlled data; or a fresh issuer/temporal audit changes performance.',
          'stop_conditions':['Risk recall <0.85 on an adequately supported labelled audit set.',
                             'Domain/language changes without validation.',
                             'Retained accuracy/coverage misses gates: revert to full review.']}
save_json('decision.json',decision)
display(Markdown(f"**Evidence-based recommendation: {choice}.** Classical macro F1 {base.macro_f1:.3f} (target ≥{D['min_macro_f1']:.2f}); risk recall {base.risk_recall:.3f} (target ≥{D['min_risk_recall']:.2f}). {decision['change_choice_if']} Human review remains mandatory for the pilot; no claim of verified environmental performance is permitted."))

## 12. Acceptance-criteria evidence and remaining work
The table below distinguishes completed evidence from checks that depend on a full GPU run. The historical transformer run supports feasibility and motivates the compute trade-off; it cannot substitute for the revised controlled comparison. **Audit-ready source and baseline evidence do not imply that every acceptance criterion has already passed.**

The final cell saves a machine-readable checklist, checksums every run file, links the run config and environment to its results, and creates a portable ZIP. Models are saved only from this run; load joblib checkpoints only from trusted sources. On Colab, download the ZIP before disconnecting because `/content` is ephemeral.

In [ ]:
audit_rows=[
 ('Reproducibility','complete','run_config.json; requirements*.txt; environment.freeze.txt; split_manifest.csv; manifest.json'),
 ('Preprocessing evidence','complete','tables/preprocessing_comparison.csv; figures/preprocessing_comparison.png'),
 ('Failure pattern → improvement','complete','tables/classical_errors.csv; tables/error_slices.csv; Section 11 annotation proposal'),
 ('Controlled comparison + validity threats','complete' if transformer_runs else 'partial — GPU comparison pending','same split_manifest.csv; paired_comparison.json on full run; Section 1 threats'),
 ('Appropriate fine-tuning + compute trade-off','complete' if transformer_runs else 'partial — historical evidence; revised training pending','models/distilbert_seed_*/trainer_state.json on full run; legacy_metrics.json; Section 6'),
 ('Explainability linked to actual uncertainty','complete' if explain_findings else 'not applicable: no observed errors','explainability_finding.json; tables/local_linear_explanation.csv; saved row IDs'),
 ('Triangulated inference','complete' if len(errors) else 'partial','confusion matrix + actual error rows + local diagnostic; triangulated_inference.md'),
 ('Decision and conditions for use','complete','decision.json; Section 11 guardrails'),
 ('Operational success criteria','complete','Section 1 proposed gates; decision.json applies the same gates'),
 ('Concise stakeholder findings and limits','complete','stakeholder_briefing.md; Section 10'),
 ('Stakeholder criteria justify choice','complete','risk recall, quality, review workload and recorded latency feed decision.json'),
 ('Risk → mitigation → check','complete','Section 11 risk/mitigation/check/decision-constraint table')]
audit=pd.DataFrame(audit_rows,columns=['criterion','status','evidence'])
save_table('acceptance_criteria',audit)
display(audit)
save_json('completion.json',{'status':'complete' if transformer_runs else 'baseline complete; controlled transformer evidence pending',
           'transformer_status':transformer_status,'run_id':RUN_ID,'config_sha256':config_hash,
           'executed_to_final_cell':True,'source_dataset_revision':CONFIG['dataset_revision'],
           'model_revision':CONFIG['model_revision'],'results_table':'tables/test_metrics.csv',
           'run_config':'run_config.json','environment':'environment.freeze.txt'})
# The checksum manifest excludes itself and the ZIP to avoid recursive hashes.
files={str(p.relative_to(RUN_DIR)):sha256(p) for p in sorted(RUN_DIR.rglob('*')) if p.is_file() and p.name!='manifest.json'}
save_json('manifest.json',{'run_id':RUN_ID,'sha256':files})
(ROOT/'artifacts/latest_run.json').write_text(json.dumps({'run_id':RUN_ID,'path':str(RUN_DIR)},indent=2)+'\n')
archive_path=shutil.make_archive(str(RUN_DIR),'zip',root_dir=RUN_DIR)
display(Markdown(f'**Saved run:** `{RUN_ID}` · [run config]({RUN_DIR.relative_to(ROOT)}/run_config.json) · [metrics]({RUN_DIR.relative_to(ROOT)}/tables/test_metrics.csv) · [manifest]({RUN_DIR.relative_to(ROOT)}/manifest.json) · [download ZIP](artifacts/{RUN_ID}.zip)'))
print('For Colab download: from google.colab import files; files.download(archive_path)')